In [1]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
import ipywidgets as widgets
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix



In [2]:
def fetch_weather(year):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":   31.1048,      
        "longitude":  77.1734,
        "start_date": f"{year}-06-01",
        "end_date":   f"{year}-08-31",
        "daily": "precipitation_sum,temperature_2m_max,wind_speed_10m_max",
        "timezone":   "Asia/Kolkata"
    }
    response = requests.get(url, params=params)
    data = response.json()["daily"]

    df = pd.DataFrame({
        "date":       data["time"],
        "rainfall":   data["precipitation_sum"],
        "temp_max":   data["temperature_2m_max"],
        "wind_speed": data["wind_speed_10m_max"]
    })
    df["year"] = year
    return df


all_weather = []
for year in range(2016, 2026):
    print(f"  Fetching {year}...", end=" ")
    all_weather.append(fetch_weather(year))
    

weather_df = pd.concat(all_weather, ignore_index=True)
weather_df["date"] = pd.to_datetime(weather_df["date"])

print(len(weather_df))
print(weather_df.head())
print(weather_df.info)

  Fetching 2016...   Fetching 2017...   Fetching 2018...   Fetching 2019...   Fetching 2020...   Fetching 2021...   Fetching 2022...   Fetching 2023...   Fetching 2024...   Fetching 2025... 920
        date  rainfall  temp_max  wind_speed  year
0 2016-06-01       0.0      23.4        16.1  2016
1 2016-06-02       0.0      24.7        17.1  2016
2 2016-06-03       0.3      24.3        15.7  2016
3 2016-06-04       0.0      26.1        15.9  2016
4 2016-06-05       0.5      25.5        13.8  2016
<bound method DataFrame.info of           date  rainfall  temp_max  wind_speed  year
0   2016-06-01       0.0      23.4        16.1  2016
1   2016-06-02       0.0      24.7        17.1  2016
2   2016-06-03       0.3      24.3        15.7  2016
3   2016-06-04       0.0      26.1        15.9  2016
4   2016-06-05       0.5      25.5        13.8  2016
..         ...       ...       ...         ...   ...
915 2025-08-27       4.5      21.5         8.1  2025
916 2025-08-28       8.6      21.5         8

In [3]:
def fetch_earthquakes(year):
   
    url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
    params = {
        "format":       "geojson",
        "starttime":    f"{year}-06-01",
        "endtime":      f"{year}-08-31",
        "minlatitude":  30.0,
        "maxlatitude":  33.5,
        "minlongitude": 75.0,
        "maxlongitude": 79.5,
        "minmagnitude": 1.5,
        "limit":        1000
    }
    response = requests.get(url, params=params)
    events = response.json()["features"]

    rows = []
    for event in events:
        rows.append({
            "date":      pd.to_datetime(event["properties"]["time"], unit="ms").normalize(),
            "magnitude": event["properties"]["mag"]
        })

    df = pd.DataFrame(rows)
    if df.empty:
        df = pd.DataFrame(columns=["date", "magnitude"])
    return df

print("Fetching earthquake data...")
all_quakes = []
for year in range(2016, 2026):
    print(f"  Fetching {year}...", end=" ")
    all_quakes.append(fetch_earthquakes(year))
    print("done")

quake_df = pd.concat(all_quakes, ignore_index=True)
print(len(quake_df))
print(quake_df.head())

Fetching earthquake data...
  Fetching 2016... done
  Fetching 2017... done
  Fetching 2018... done
  Fetching 2019... done
  Fetching 2020... done
  Fetching 2021... done
  Fetching 2022... done
  Fetching 2023... done
  Fetching 2024... done
  Fetching 2025... done
33
        date  magnitude
0 2016-08-27        4.1
1 2016-08-27        4.5
2 2016-08-27        4.6
3 2016-08-01        4.5
4 2017-08-16        4.4


In [4]:

if not quake_df.empty:
    daily_quake = (
        quake_df.groupby("date")["magnitude"]
        .max()
        .reset_index()
        .rename(columns={"magnitude": "seismic_activity"})
    )
else:
    daily_quake = pd.DataFrame(columns=["date", "seismic_activity"])


final_df = weather_df.merge(daily_quake, on="date", how="left")
final_df["seismic_activity"] = final_df["seismic_activity"].fillna(0)


for col in ["rainfall", "temp_max", "wind_speed"]:
    final_df[col] = final_df[col].fillna(final_df[col].median())


final_df = final_df.sort_values("date").reset_index(drop=True)
final_df["rain_2day"] = final_df["rainfall"].rolling(window=2, min_periods=1).sum()

print("✅ Final dataset shape:", final_df.shape)
print("   Columns:", final_df.columns.tolist())
print()
print(final_df.head())

✅ Final dataset shape: (920, 7)
   Columns: ['date', 'rainfall', 'temp_max', 'wind_speed', 'year', 'seismic_activity', 'rain_2day']

        date  rainfall  temp_max  wind_speed  year  seismic_activity  \
0 2016-06-01       0.0      23.4        16.1  2016               0.0   
1 2016-06-02       0.0      24.7        17.1  2016               0.0   
2 2016-06-03       0.3      24.3        15.7  2016               0.0   
3 2016-06-04       0.0      26.1        15.9  2016               0.0   
4 2016-06-05       0.5      25.5        13.8  2016               0.0   

   rain_2day  
0        0.0  
1        0.0  
2        0.3  
3        0.3  
4        0.5  


In [5]:

max_rainfall  = final_df["rainfall"].max()
max_rain_2day = final_df["rain_2day"].max()
max_wind      = final_df["wind_speed"].max()
max_seismic   = max(final_df["seismic_activity"].max(), 1)  

final_df["risk_score"] = (
    (final_df["rainfall"]         / max_rainfall)  * 40 +
    (final_df["rain_2day"]        / max_rain_2day) * 20 +
    (final_df["wind_speed"]       / max_wind)      * 25 +
    (final_df["seismic_activity"] / max_seismic)   * 15
)


p_high   = final_df["risk_score"].quantile(0.80)   
p_medium = final_df["risk_score"].quantile(0.45)   

def label_risk(score):
    if score >= p_high:
        return "High"
    elif score >= p_medium:
        return "Medium"
    else:
        return "Low"

final_df["Risk_Level"] = final_df["risk_score"].apply(label_risk)

print("Risk Level Distribution:")
print(final_df["Risk_Level"].value_counts())
print()
print(f"Percentile cutoffs used:")
print(f"  High   ≥ {p_high:.2f}  (top 20% of all days)")
print(f"  Medium ≥ {p_medium:.2f}  (45th–80th percentile)")
print(f"  Low    <  {p_medium:.2f}  (bottom 45%)")

Risk Level Distribution:
Risk_Level
Low       414
Medium    322
High      184
Name: count, dtype: int64

Percentile cutoffs used:
  High   ≥ 17.09  (top 20% of all days)
  Medium ≥ 13.59  (45th–80th percentile)
  Low    <  13.59  (bottom 45%)


In [6]:
input_features = ["rainfall", "rain_2day", "temp_max", "wind_speed", "seismic_activity"]
X = final_df[input_features]
y = final_df["Risk_Level"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Class counts in training set:")
print(y_train.value_counts())
print()
print("Class counts in test set:")
print(y_test.value_counts())


model = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy    = accuracy_score(y_test, predictions)


print(f"\n  Accuracy: {accuracy * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, predictions))

Class counts in training set:
Risk_Level
Low       331
Medium    258
High      147
Name: count, dtype: int64

Class counts in test set:
Risk_Level
Low       83
Medium    64
High      37
Name: count, dtype: int64

  Accuracy: 89.67%

Classification Report:
              precision    recall  f1-score   support

        High       0.97      0.81      0.88        37
         Low       0.95      0.92      0.93        83
      Medium       0.81      0.92      0.86        64

    accuracy                           0.90       184
   macro avg       0.91      0.88      0.89       184
weighted avg       0.90      0.90      0.90       184



In [7]:


risk_colors = {"High": "#e74c3c", "Medium": "#e67e22", "Low": "#27ae60"}

def plot_dashboard(year):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(
        f"Multi-Hazard Risk Prediction\nHimachal Pradesh Monsoon Data (2016–2025, Daily) — Year: {year}",
        fontsize=14, fontweight="bold"
    )

    data_year = final_df[final_df["year"] == year].copy().reset_index(drop=True)
    bar_colors = [risk_colors[r] for r in data_year["Risk_Level"]]

    axes[0, 0].bar(range(len(data_year)), data_year["rainfall"],
                   color=bar_colors, alpha=0.85, width=0.8)
    axes[0, 0].axhline(64.5, color="#e74c3c", linestyle="--",
                       linewidth=1.5, label="IMD Heavy Rain (64.5mm)")
    axes[0, 0].axhline(15.6, color="#e67e22", linestyle="--",
                       linewidth=1.5, label="IMD Moderate Rain (15.6mm)")
    legend_patches = [
        Patch(color="#e74c3c", label="High Risk"),
        Patch(color="#e67e22", label="Medium Risk"),
        Patch(color="#27ae60", label="Low Risk")
    ]
    axes[0, 0].legend(handles=legend_patches, fontsize=8, loc="upper right")
    axes[0, 0].set_title(f"Daily Rainfall — Monsoon {year}\n(Bar color = risk level)")
    axes[0, 0].set_xlabel("Day of Monsoon Season (Jun 1 → Aug 31)")
    axes[0, 0].set_ylabel("Total Rainfall (mm/day)")

    risk_counts = final_df[final_df["year"] == year]["Risk_Level"].value_counts()
    present_levels = [l for l in risk_counts.index if l in risk_colors]
    axes[0, 1].pie(
        risk_counts[present_levels].values,
        labels=present_levels,
        colors=[risk_colors[l] for l in present_levels],
        autopct="%1.1f%%",
        startangle=90,
        wedgeprops={"edgecolor": "white", "linewidth": 2}
    )
    axes[0, 1].set_title(f"Risk Level Distribution — {year}")

    feat_imp = pd.Series(model.feature_importances_, index=input_features)
    feat_imp_sorted = feat_imp.sort_values(ascending=True)
    max_imp = feat_imp_sorted.max()
    bar_c = [
        "#e74c3c" if v >= max_imp * 0.75 else
        "#e67e22" if v >= max_imp * 0.45 else
        "#3498db"
        for v in feat_imp_sorted.values
    ]
    feat_imp_sorted.plot(kind="barh", ax=axes[1, 0], color=bar_c, edgecolor="white")
    axes[1, 0].set_title("Feature Importance\n(Which factor drives risk the most?)")
    axes[1, 0].set_xlabel("Importance Score")
    for i, v in enumerate(feat_imp_sorted.values):
        axes[1, 0].text(v + 0.002, i, f"{v:.3f}", va="center", fontsize=9)

    year_df = final_df[final_df["year"] == year]
    X_year = year_df[input_features]
    y_year = year_df["Risk_Level"]
    preds_year = model.predict(X_year)
    acc_year = (preds_year == y_year).mean()
    cm = confusion_matrix(y_year, preds_year, labels=["High", "Medium", "Low"])
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["High", "Medium", "Low"],
        yticklabels=["High", "Medium", "Low"],
        ax=axes[1, 1], linewidths=0.5,
        annot_kws={"size": 13, "weight": "bold"}
    )
    axes[1, 1].set_title(f"Confusion Matrix  (Accuracy: {acc_year*100:.2f}%)")
    axes[1, 1].set_xlabel("Predicted")
    axes[1, 1].set_ylabel("Actual")

    plt.tight_layout()
    plt.show()

year_slider = widgets.IntSlider(
    value=final_df["year"].min(),
    min=final_df["year"].min(),
    max=final_df["year"].max(),
    step=1,
    description="Year:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

widgets.interact(plot_dashboard, year=year_slider)

interactive(children=(IntSlider(value=2016, description='Year:', layout=Layout(width='400px'), max=2025, min=2…

<function __main__.plot_dashboard(year)>

In [8]:
pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Ayaan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [9]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# ── COLOR SCHEME ──────────────────────────────────────────────
BG        = "#0d1117"
PANEL_BG  = "#161b22"
GRID_COL  = "#21262d"
TEXT      = "#e6edf3"
DIM       = "#8b949e"
HIGH      = "#ff4d4d"
MED       = "#ffa500"
LOW       = "#00c853"
ACCENT    = "#58a6ff"

RISK_COLORS = {"High": HIGH, "Medium": MED, "Low": LOW}

plt.rcParams.update({
    "figure.facecolor":  BG,
    "axes.facecolor":    PANEL_BG,
    "axes.edgecolor":    GRID_COL,
    "axes.labelcolor":   DIM,
    "axes.titlecolor":   TEXT,
    "axes.grid":         True,
    "grid.color":        GRID_COL,
    "grid.linewidth":    0.6,
    "xtick.color":       DIM,
    "ytick.color":       DIM,
    "text.color":        TEXT,
    "font.family":       "monospace",
    "legend.facecolor":  PANEL_BG,
    "legend.edgecolor":  GRID_COL,
})

# ── DASHBOARD ─────────────────────────────────────────────────
def plot_dashboard(year):
    data = final_df[final_df["year"] == year].copy().reset_index(drop=True)

    if data.empty:
        print(f"No data for {year}")
        return

    days        = np.arange(len(data))
    rainfall    = data["rainfall"].values
    risk_levels = data["Risk_Level"].values
    bar_colors  = [RISK_COLORS[r] for r in risk_levels]

    counts      = data["Risk_Level"].value_counts()
    high_pct    = counts.get("High",   0) / len(data) * 100
    med_pct     = counts.get("Medium", 0) / len(data) * 100
    low_pct     = counts.get("Low",    0) / len(data) * 100
    avg_rain    = rainfall.mean()
    max_rain    = rainfall.max()

    fig = plt.figure(figsize=(18, 13), facecolor=BG)
    fig.suptitle(
        f"SafeSphere  ·  Himachal Pradesh Monsoon Intelligence  ·  {year}",
        fontsize=15, fontweight="bold", color=TEXT, y=0.97
    )

    gs = gridspec.GridSpec(
        2, 3,
        figure=fig,
        hspace=0.45, wspace=0.35,
        top=0.91, bottom=0.07,
        left=0.06, right=0.97
    )

    # ── 1. DAILY RAINFALL BAR (spans full top row) ──────────────
    ax1 = fig.add_subplot(gs[0, :])
    ax1.bar(days, rainfall, color=bar_colors, alpha=0.9, width=0.85, zorder=3)
    ax1.fill_between(days, rainfall, alpha=0.08, color=ACCENT)
    ax1.axhline(64.5, color=HIGH,  lw=1.5, ls="--", zorder=4)
    ax1.axhline(15.6, color=MED,   lw=1.5, ls="--", zorder=4)
    ax1.text(len(days)+0.5, 64.5, "Heavy\n64.5mm",  color=HIGH, fontsize=7.5, va="center")
    ax1.text(len(days)+0.5, 15.6, "Moderate\n15.6mm", color=MED,  fontsize=7.5, va="center")
    ax1.set_title(f"Daily Rainfall  (Jun 1 → Aug 31, {year})  ·  Avg: {avg_rain:.1f} mm  ·  Peak: {max_rain:.1f} mm",
                  fontsize=11, pad=10)
    ax1.set_xlabel("Day of Monsoon Season", fontsize=9)
    ax1.set_ylabel("Rainfall (mm/day)", fontsize=9)
    ax1.set_xlim(-1, len(days) + 8)
    ax1.set_ylim(0, max(rainfall.max() * 1.18, 80))
    patches = [
        mpatches.Patch(color=HIGH, label="High Risk"),
        mpatches.Patch(color=MED,  label="Medium Risk"),
        mpatches.Patch(color=LOW,  label="Low Risk"),
    ]
    ax1.legend(handles=patches, fontsize=8.5, loc="upper left",
               framealpha=0.3, ncol=3)

    # ── 2. 3D BAR: WEEKLY RAINFALL ───────────────────────────────
    ax2 = fig.add_subplot(gs[1, 0], projection="3d")
    ax2.set_facecolor(PANEL_BG)

    n_weeks   = len(days) // 7
    week_rain = [rainfall[i*7:(i+1)*7].sum() for i in range(n_weeks)]
    week_risk = []
    for i in range(n_weeks):
        seg = risk_levels[i*7:(i+1)*7]
        week_risk.append("High" if "High" in seg else ("Medium" if "Medium" in seg else "Low"))

    xpos  = np.arange(n_weeks)
    ypos  = np.zeros(n_weeks)
    zpos  = np.zeros(n_weeks)
    dx = dy = 0.6
    dz    = np.array(week_rain)
    wcols = [RISK_COLORS[r] for r in week_risk]

    for xi, yi, zi, dzi, c in zip(xpos, ypos, zpos, dz, wcols):
        ax2.bar3d(xi, yi, zi, dx, dy, dzi, color=c, alpha=0.82, shade=True)

    ax2.set_title("Weekly Rainfall\n(3D by Risk)", fontsize=9, pad=6)
    ax2.set_xlabel("Week", fontsize=7, labelpad=4)
    ax2.set_zlabel("mm", fontsize=7, labelpad=2)
    ax2.set_xticks(xpos + 0.3)
    ax2.set_xticklabels([f"W{i+1}" for i in range(n_weeks)], fontsize=6.5)
    ax2.tick_params(colors=DIM, labelsize=6)
    ax2.xaxis.pane.fill = False
    ax2.yaxis.pane.fill = False
    ax2.zaxis.pane.fill = False
    ax2.xaxis.pane.set_edgecolor(GRID_COL)
    ax2.yaxis.pane.set_edgecolor(GRID_COL)
    ax2.zaxis.pane.set_edgecolor(GRID_COL)
    ax2.view_init(elev=28, azim=-55)

    # ── 3. DONUT: RISK DISTRIBUTION ──────────────────────────────
    ax3 = fig.add_subplot(gs[1, 1])
    labels = [k for k in ["High","Medium","Low"] if k in counts]
    vals   = [counts[k] for k in labels]
    cols   = [RISK_COLORS[k] for k in labels]
    wedges, texts, autos = ax3.pie(
        vals, labels=None, colors=cols,
        autopct="%1.1f%%", startangle=90,
        pctdistance=0.78,
        wedgeprops={"width": 0.52, "edgecolor": BG, "linewidth": 2.5}
    )
    for at in autos:
        at.set_fontsize(9)
        at.set_color(TEXT)
        at.set_fontweight("bold")
    ax3.text(0, 0, f"{len(data)}\ndays", ha="center", va="center",
             fontsize=11, fontweight="bold", color=TEXT)
    ax3.set_title(f"Risk Distribution\n{year}", fontsize=9, pad=8)
    ax3.legend(
        handles=[mpatches.Patch(color=RISK_COLORS[l], label=f"{l}  ({counts.get(l,0)}d)") for l in ["High","Medium","Low"]],
        fontsize=7.5, loc="lower center", ncol=3,
        bbox_to_anchor=(0.5, -0.12), framealpha=0.2
    )

    # ── 4. ROLLING AVERAGE LINE ───────────────────────────────────
    ax4 = fig.add_subplot(gs[1, 2])
    roll7  = pd.Series(rainfall).rolling(7,  min_periods=1).mean().values
    roll14 = pd.Series(rainfall).rolling(14, min_periods=1).mean().values

    ax4.fill_between(days, rainfall, alpha=0.1, color=ACCENT)
    ax4.plot(days, rainfall,  color=ACCENT,  lw=0.8,  alpha=0.4,  label="Daily")
    ax4.plot(days, roll7,     color=LOW,     lw=1.8,  alpha=0.9,  label="7-day avg")
    ax4.plot(days, roll14,    color=MED,     lw=1.8,  alpha=0.9,  label="14-day avg", ls="--")
    ax4.axhline(64.5, color=HIGH, lw=1, ls=":", alpha=0.6)
    ax4.set_title("Rolling Rainfall Trend", fontsize=9, pad=8)
    ax4.set_xlabel("Day", fontsize=8)
    ax4.set_ylabel("mm/day", fontsize=8)
    ax4.legend(fontsize=7.5, framealpha=0.3, loc="upper right")
    ax4.set_xlim(0, len(days)-1)
    ax4.set_ylim(0, max(rainfall.max() * 1.18, 80))

    plt.show()


# ── WIDGET ────────────────────────────────────────────────────
years = sorted(final_df["year"].unique().tolist())

year_slider = widgets.SelectionSlider(
    options=years,
    value=years[-1],
    description="📅 Year:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="480px")
)

year_label = widgets.HTML(
    value=f"<b style='color:#58a6ff;font-size:15px'>{years[-1]}</b>",
)

def on_change(change):
    year_label.value = f"<b style='color:#58a6ff;font-size:15px'>{change['new']}</b>"

year_slider.observe(on_change, names="value")

out = widgets.interactive_output(plot_dashboard, {"year": year_slider})

header = widgets.HTML("""
<div style='
  background:#161b22;
  border:1px solid #21262d;
  border-radius:10px;
  padding:12px 20px;
  margin-bottom:10px;
  font-family:monospace;
'>
  <span style='color:#58a6ff;font-size:13px;letter-spacing:2px'>
    🛡 SAFESPHERE  ·  HIMACHAL PRADESH MONSOON INTELLIGENCE
  </span><br>
  <span style='color:#8b949e;font-size:11px'>
    Drag the slider to explore any monsoon season · 3 views update simultaneously
  </span>
</div>
""")

controls = widgets.HBox(
    [widgets.Label("Select Season →", style={"font_weight": "bold"}), year_slider, year_label],
    layout=widgets.Layout(
        align_items="center", gap="12px",
        padding="10px 16px",
        border="1px solid #21262d",
        border_radius="8px",
        margin="0 0 14px 0"
    )
)

display(header, controls, out)

HTML(value="\n<div style='\n  background:#161b22;\n  border:1px solid #21262d;\n  border-radius:10px;\n  paddi…

Output()

In [ ]:

import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from mpl_toolkits.mplot3d import Axes3D


PANEL_BG  = "#161b22"
GRID_COL  = "#21262d"
TEXT      = "#e6edf3"
DIM       = "#8b949e"
HIGH      = "#ff4d4d"
MED       = "#ffa500"
LOW       = "#00c853"
ACCENT    = "#58a6ff"

RISK_COLORS = {"High": HIGH, "Medium": MED, "Low": LOW}

plt.rcParams.update({
    "figure.facecolor":  BG,
    "axes.facecolor":    PANEL_BG,
    "axes.edgecolor":    GRID_COL,
    "axes.labelcolor":   DIM,
    "axes.titlecolor":   TEXT,
    "axes.grid":         True,
    "grid.color":        GRID_COL,
    "grid.linewidth":    0.6,
    "xtick.color":       DIM,
    "ytick.color":       DIM,
    "text.color":        TEXT,
    "font.family":       "monospace",
    "legend.facecolor":  PANEL_BG,
    "legend.edgecolor":  GRID_COL,
})


def plot_dashboard(year):
    data = final_df[final_df["year"] == year].copy().reset_index(drop=True)

    if data.empty:
        print(f"No data for {year}")
        return

    days        = np.arange(len(data))
    rainfall    = data["rainfall"].values
    risk_levels = data["Risk_Level"].values
    bar_colors  = [RISK_COLORS[r] for r in risk_levels]

    counts      = data["Risk_Level"].value_counts()
    high_pct    = counts.get("High",   0) / len(data) * 100
    med_pct     = counts.get("Medium", 0) / len(data) * 100
    low_pct     = counts.get("Low",    0) / len(data) * 100
    avg_rain    = rainfall.mean()
    max_rain    = rainfall.max()

    fig = plt.figure(figsize=(18, 13), facecolor=BG)
    fig.suptitle(
        f"SafeSphere  ·  Himachal Pradesh Monsoon Intelligence  ·  {year}",
        fontsize=15, fontweight="bold", color=TEXT, y=0.97
    )

    gs = gridspec.GridSpec(
        2, 3,
        figure=fig,
        hspace=0.45, wspace=0.35,
        top=0.91, bottom=0.07,
        left=0.06, right=0.97
    )
    ax1 = fig.add_subplot(gs[0, :])
    ax1.bar(days, rainfall, color=bar_colors, alpha=0.9, width=0.85, zorder=3)
    ax1.fill_between(days, rainfall, alpha=0.08, color=ACCENT)
    ax1.axhline(64.5, color=HIGH,  lw=1.5, ls="--", zorder=4)
    ax1.axhline(15.6, color=MED,   lw=1.5, ls="--", zorder=4)
    ax1.text(len(days)+0.5, 64.5, "Heavy\n64.5mm",  color=HIGH, fontsize=7.5, va="center")
    ax1.text(len(days)+0.5, 15.6, "Moderate\n15.6mm", color=MED,  fontsize=7.5, va="center")
    ax1.set_title(f"Daily Rainfall  (Jun 1 → Aug 31, {year})  ·  Avg: {avg_rain:.1f} mm  ·  Peak: {max_rain:.1f} mm",
                  fontsize=11, pad=10)
    ax1.set_xlabel("Day of Monsoon Season", fontsize=9)
    ax1.set_ylabel("Rainfall (mm/day)", fontsize=9)
    ax1.set_xlim(-1, len(days) + 8)
    ax1.set_ylim(0, max(rainfall.max() * 1.18, 80))
    patches = [
        mpatches.Patch(color=HIGH, label="High Risk"),
        mpatches.Patch(color=MED,  label="Medium Risk"),
        mpatches.Patch(color=LOW,  label="Low Risk"),
    ]
    ax1.legend(handles=patches, fontsize=8.5, loc="upper left",
               framealpha=0.3, ncol=3)

    ax2 = fig.add_subplot(gs[1, 0], projection="3d")
    ax2.set_facecolor(PANEL_BG)

    n_weeks   = len(days) // 7
    week_rain = [rainfall[i*7:(i+1)*7].sum() for i in range(n_weeks)]
    week_risk = []
    for i in range(n_weeks):
        seg = risk_levels[i*7:(i+1)*7]
        week_risk.append("High" if "High" in seg else ("Medium" if "Medium" in seg else "Low"))

    xpos  = np.arange(n_weeks)
    ypos  = np.zeros(n_weeks)
    zpos  = np.zeros(n_weeks)
    dx = dy = 0.6
    dz    = np.array(week_rain)
    wcols = [RISK_COLORS[r] for r in week_risk]

    for xi, yi, zi, dzi, c in zip(xpos, ypos, zpos, dz, wcols):
        ax2.bar3d(xi, yi, zi, dx, dy, dzi, color=c, alpha=0.82, shade=True)

    ax2.set_title("Weekly Rainfall\n(3D by Risk)", fontsize=9, pad=6)
    ax2.set_xlabel("Week", fontsize=7, labelpad=4)
    ax2.set_zlabel("mm", fontsize=7, labelpad=2)
    ax2.set_xticks(xpos + 0.3)
    ax2.set_xticklabels([f"W{i+1}" for i in range(n_weeks)], fontsize=6.5)
    ax2.tick_params(colors=DIM, labelsize=6)
    ax2.xaxis.pane.fill = False
    ax2.yaxis.pane.fill = False
    ax2.zaxis.pane.fill = False
    ax2.xaxis.pane.set_edgecolor(GRID_COL)
    ax2.yaxis.pane.set_edgecolor(GRID_COL)
    ax2.zaxis.pane.set_edgecolor(GRID_COL)
    ax2.view_init(elev=28, azim=-55)

    ax3 = fig.add_subplot(gs[1, 1])
    labels = [k for k in ["High","Medium","Low"] if k in counts]
    vals   = [counts[k] for k in labels]
    cols   = [RISK_COLORS[k] for k in labels]
    wedges, texts, autos = ax3.pie(
        vals, labels=None, colors=cols,
        autopct="%1.1f%%", startangle=90,
        pctdistance=0.78,
        wedgeprops={"width": 0.52, "edgecolor": BG, "linewidth": 2.5}
    )
    for at in autos:
        at.set_fontsize(9)
        at.set_color(TEXT)
        at.set_fontweight("bold")
    ax3.text(0, 0, f"{len(data)}\ndays", ha="center", va="center",
             fontsize=11, fontweight="bold", color=TEXT)
    ax3.set_title(f"Risk Distribution\n{year}", fontsize=9, pad=8)
    ax3.legend(
        handles=[mpatches.Patch(color=RISK_COLORS[l], label=f"{l}  ({counts.get(l,0)}d)") for l in ["High","Medium","Low"]],
        fontsize=7.5, loc="lower center", ncol=3,
        bbox_to_anchor=(0.5, -0.12), framealpha=0.2
    )

    ax4 = fig.add_subplot(gs[1, 2])
    roll7  = pd.Series(rainfall).rolling(7,  min_periods=1).mean().values
    roll14 = pd.Series(rainfall).rolling(14, min_periods=1).mean().values

    ax4.fill_between(days, rainfall, alpha=0.1, color=ACCENT)
    ax4.plot(days, rainfall,  color=ACCENT,  lw=0.8,  alpha=0.4,  label="Daily")
    ax4.plot(days, roll7,     color=LOW,     lw=1.8,  alpha=0.9,  label="7-day avg")
    ax4.plot(days, roll14,    color=MED,     lw=1.8,  alpha=0.9,  label="14-day avg", ls="--")
    ax4.axhline(64.5, color=HIGH, lw=1, ls=":", alpha=0.6)
    ax4.set_title("Rolling Rainfall Trend", fontsize=9, pad=8)
    ax4.set_xlabel("Day", fontsize=8)
    ax4.set_ylabel("mm/day", fontsize=8)
    ax4.legend(fontsize=7.5, framealpha=0.3, loc="upper right")
    ax4.set_xlim(0, len(days)-1)
    ax4.set_ylim(0, max(rainfall.max() * 1.18, 80))

    plt.show()



years = sorted(final_df["year"].unique().tolist())

year_slider = widgets.SelectionSlider(
    options=years,
    value=years[-1],
    description="📅 Year:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="480px")
)

year_label = widgets.HTML(
    value=f"<b style='color:#58a6ff;font-size:15px'>{years[-1]}</b>",
)

def on_change(change):
    year_label.value = f"<b style='color:#58a6ff;font-size:15px'>{change['new']}</b>"

year_slider.observe(on_change, names="value")

out = widgets.interactive_output(plot_dashboard, {"year": year_slider})

header = widgets.HTML("""
<div style='
  background:#161b22;
  border:1px solid #21262d;
  border-radius:10px;
  padding:12px 20px;
  margin-bottom:10px;
  font-family:monospace;
'>
  <span style='color:#58a6ff;font-size:13px;letter-spacing:2px'>
    🛡 SAFESPHERE  ·  HIMACHAL PRADESH MONSOON INTELLIGENCE
  </span><br>
  <span style='color:#8b949e;font-size:11px'>
    Drag the slider to explore any monsoon season · 3 views update simultaneously
  </span>
</div>
""")

controls = widgets.HBox(
    [widgets.Label("Select Season →", style={"font_weight": "bold"}), year_slider, year_label],
    layout=widgets.Layout(
        align_items="center", gap="12px",
        padding="10px 16px",
        border="1px solid #21262d",
        border_radius="8px",
        margin="0 0 14px 0"
    )
)

display(header, controls, out)

HTML(value="\n<div style='\n  background:#161b22;\n  border:1px solid #21262d;\n  border-radius:10px;\n  paddi…

Output()